In [1]:
# Test yfinance is working correctly
import yfinance as yf

ticker = "AAPL"
stock = yf.Ticker(ticker)

# Test 1 — company info
info = stock.info
print("Company Name:", info.get("longName"))
print("Sector:", info.get("sector"))
print("Market Cap:", info.get("marketCap"))

Company Name: Apple Inc.
Sector: Technology
Market Cap: 4525762019328


In [3]:
# Test financial statements
import pandas as pd

# Annual income statement
annual_income = stock.financials
print("ANNUAL INCOME STATEMENT")
print(annual_income)

ANNUAL INCOME STATEMENT
                                                      2025-09-30  \
Tax Effect Of Unusual Items                         0.000000e+00   
Tax Rate For Calcs                                  1.560000e-01   
Normalized EBITDA                                   1.447480e+11   
Net Income From Continuing Operation Net Minori...  1.120100e+11   
Reconciled Depreciation                             1.169800e+10   
Reconciled Cost Of Revenue                          2.209600e+11   
EBITDA                                              1.447480e+11   
EBIT                                                1.330500e+11   
Net Interest Income                                          NaN   
Interest Expense                                             NaN   
Interest Income                                              NaN   
Normalized Income                                   1.120100e+11   
Net Income From Continuing And Discontinued Ope...  1.120100e+11   
Total Expenses          

In [4]:
# Test balance sheet and cash flow
annual_balance = stock.balance_sheet
annual_cashflow = stock.cashflow

print("BALANCE SHEET ROWS:")
print(annual_balance.index.tolist())

print("\nCASH FLOW ROWS:")
print(annual_cashflow.index.tolist())

BALANCE SHEET ROWS:
['Treasury Shares Number', 'Ordinary Shares Number', 'Share Issued', 'Net Debt', 'Total Debt', 'Tangible Book Value', 'Invested Capital', 'Working Capital', 'Net Tangible Assets', 'Capital Lease Obligations', 'Common Stock Equity', 'Total Capitalization', 'Total Equity Gross Minority Interest', 'Stockholders Equity', 'Gains Losses Not Affecting Retained Earnings', 'Other Equity Adjustments', 'Retained Earnings', 'Capital Stock', 'Common Stock', 'Total Liabilities Net Minority Interest', 'Total Non Current Liabilities Net Minority Interest', 'Other Non Current Liabilities', 'Tradeand Other Payables Non Current', 'Long Term Debt And Capital Lease Obligation', 'Long Term Capital Lease Obligation', 'Long Term Debt', 'Current Liabilities', 'Other Current Liabilities', 'Current Deferred Liabilities', 'Current Deferred Revenue', 'Current Debt And Capital Lease Obligation', 'Current Capital Lease Obligation', 'Current Debt', 'Other Current Borrowings', 'Commercial Paper', '

In [5]:
# Extract the specific fields we need and check for NaN
import numpy as np

# Most recent year column (first column)
latest_year = annual_income.columns[0]

# Fields to extract from each statement
fields_income = ['EBIT', 'EBITDA', 'Total Revenue', 'Interest Expense', 'Net Income']
fields_balance = ['Total Debt', 'Net Debt', 'Cash And Cash Equivalents', 
                  'Total Assets', 'Current Assets', 'Current Liabilities', 
                  'Working Capital', 'Retained Earnings']
fields_cashflow = ['Operating Cash Flow', 'Free Cash Flow', 
                   'Capital Expenditure', 'Interest Paid Supplemental Data']

print("=== INCOME STATEMENT (Latest Year) ===")
for field in fields_income:
    try:
        value = annual_income.loc[field, latest_year]
        print(f"{field}: {value:,.0f}" if not pd.isna(value) else f"{field}: NOT AVAILABLE")
    except KeyError:
        print(f"{field}: FIELD MISSING")

print("\n=== BALANCE SHEET (Latest Year) ===")
for field in fields_balance:
    try:
        value = annual_balance.loc[field, latest_year]
        print(f"{field}: {value:,.0f}" if not pd.isna(value) else f"{field}: NOT AVAILABLE")
    except KeyError:
        print(f"{field}: FIELD MISSING")

print("\n=== CASH FLOW (Latest Year) ===")
for field in fields_cashflow:
    try:
        value = annual_cashflow.loc[field, latest_year]
        print(f"{field}: {value:,.0f}" if not pd.isna(value) else f"{field}: NOT AVAILABLE")
    except KeyError:
        print(f"{field}: FIELD MISSING")

=== INCOME STATEMENT (Latest Year) ===
EBIT: 133,050,000,000
EBITDA: 144,748,000,000
Total Revenue: 416,161,000,000
Interest Expense: NOT AVAILABLE
Net Income: 112,010,000,000

=== BALANCE SHEET (Latest Year) ===
Total Debt: 98,657,000,000
Net Debt: 62,723,000,000
Cash And Cash Equivalents: 35,934,000,000
Total Assets: 359,241,000,000
Current Assets: 147,957,000,000
Current Liabilities: 165,631,000,000
Working Capital: -17,674,000,000
Retained Earnings: -14,264,000,000

=== CASH FLOW (Latest Year) ===
Operating Cash Flow: 111,482,000,000
Free Cash Flow: 98,767,000,000
Capital Expenditure: -12,715,000,000
Interest Paid Supplemental Data: NOT AVAILABLE


In [6]:
# Handle Interest Expense fallback + compute all ratios

def get_interest_expense(income_stmt):
    """
    Try to get interest expense from multiple possible fields.
    Falls back across years if latest year is NaN.
    """
    # Try direct Interest Expense first
    for col in income_stmt.columns:
        try:
            val = income_stmt.loc['Interest Expense', col]
            if not pd.isna(val) and val != 0:
                print(f"Interest Expense found in {col}: {val:,.0f}")
                return abs(val)  # make positive
        except KeyError:
            break
    
    # Try Interest Expense Non Operating
    for col in income_stmt.columns:
        try:
            val = income_stmt.loc['Interest Expense Non Operating', col]
            if not pd.isna(val) and val != 0:
                print(f"Interest Expense Non Operating found in {col}: {val:,.0f}")
                return abs(val)
        except KeyError:
            break
    
    print("Interest Expense: NOT FOUND in any year")
    return None

# Get latest year values
latest_year = annual_income.columns[0]

def safe_get(df, field, col):
    """Safely extract a field, returning None if missing or NaN."""
    try:
        val = df.loc[field, col]
        return float(val) if not pd.isna(val) else None
    except KeyError:
        return None

# Extract all fields
ebit        = safe_get(annual_income, 'EBIT', latest_year)
ebitda      = safe_get(annual_income, 'EBITDA', latest_year)
revenue     = safe_get(annual_income, 'Total Revenue', latest_year)
net_income  = safe_get(annual_income, 'Net Income', latest_year)
total_debt  = safe_get(annual_balance, 'Total Debt', latest_year)
net_debt    = safe_get(annual_balance, 'Net Debt', latest_year)
cash        = safe_get(annual_balance, 'Cash And Cash Equivalents', latest_year)
total_assets = safe_get(annual_balance, 'Total Assets', latest_year)
current_assets = safe_get(annual_balance, 'Current Assets', latest_year)
current_liabilities = safe_get(annual_balance, 'Current Liabilities', latest_year)
working_capital = safe_get(annual_balance, 'Working Capital', latest_year)
retained_earnings = safe_get(annual_balance, 'Retained Earnings', latest_year)
operating_cf = safe_get(annual_cashflow, 'Operating Cash Flow', latest_year)
free_cf      = safe_get(annual_cashflow, 'Free Cash Flow', latest_year)
capex        = safe_get(annual_cashflow, 'Capital Expenditure', latest_year)
interest_expense = get_interest_expense(annual_income)

# Now compute ratios
print("\n=== COMPUTED RATIOS ===")

# 1. Debt / EBITDA
if total_debt and ebitda:
    debt_to_ebitda = total_debt / ebitda
    print(f"Debt/EBITDA: {debt_to_ebitda:.2f}x")
else:
    debt_to_ebitda = None
    print("Debt/EBITDA: NOT AVAILABLE")

# 2. Interest Coverage (EBIT / Interest Expense)
if ebit and interest_expense:
    interest_coverage = ebit / interest_expense
    print(f"Interest Coverage: {interest_coverage:.2f}x")
else:
    interest_coverage = None
    print("Interest Coverage: NOT AVAILABLE — Interest Expense missing")

# 3. Current Ratio
if current_assets and current_liabilities:
    current_ratio = current_assets / current_liabilities
    print(f"Current Ratio: {current_ratio:.2f}x")
else:
    current_ratio = None
    print("Current Ratio: NOT AVAILABLE")

# 4. Free Cash Flow to Debt
if free_cf and total_debt:
    fcf_to_debt = free_cf / total_debt
    print(f"FCF to Debt: {fcf_to_debt:.2f}x")
else:
    fcf_to_debt = None
    print("FCF to Debt: NOT AVAILABLE")

# 5. Net Debt / EBITDA
if net_debt and ebitda:
    net_debt_to_ebitda = net_debt / ebitda
    print(f"Net Debt/EBITDA: {net_debt_to_ebitda:.2f}x")
else:
    net_debt_to_ebitda = None
    print("Net Debt/EBITDA: NOT AVAILABLE")

# 6. DSCR (Operating CF / Total Debt)
# Using Operating CF / Total Debt as proxy since current debt service not always available
if operating_cf and total_debt:
    dscr = operating_cf / total_debt
    print(f"DSCR (proxy): {dscr:.2f}x")
else:
    dscr = None
    print("DSCR: NOT AVAILABLE")

# 7. Altman Z-Score
# Z = 1.2*(WC/TA) + 1.4*(RE/TA) + 3.3*(EBIT/TA) + 0.6*(MktCap/TotalDebt) + 1.0*(Rev/TA)
market_cap = stock.info.get('marketCap')
if all([working_capital, retained_earnings, ebit, market_cap, total_debt, revenue, total_assets]):
    z1 = 1.2 * (working_capital / total_assets)
    z2 = 1.4 * (retained_earnings / total_assets)
    z3 = 3.3 * (ebit / total_assets)
    z4 = 0.6 * (market_cap / total_debt)
    z5 = 1.0 * (revenue / total_assets)
    altman_z = z1 + z2 + z3 + z4 + z5
    if altman_z > 2.99:
        zone = "SAFE"
    elif altman_z > 1.81:
        zone = "GREY"
    else:
        zone = "DISTRESS"
    print(f"Altman Z-Score: {altman_z:.2f} ({zone})")
else:
    altman_z = None
    print("Altman Z-Score: NOT AVAILABLE")

Interest Expense found in 2023-09-30 00:00:00: 3,933,000,000

=== COMPUTED RATIOS ===
Debt/EBITDA: 0.68x
Interest Coverage: 33.83x
Current Ratio: 0.89x
FCF to Debt: 1.00x
Net Debt/EBITDA: 0.43x
DSCR (proxy): 1.13x
Altman Z-Score: 29.79 (SAFE)


In [7]:
def get_financial_ratios(ticker: str) -> dict:
    """
    Pulls financial statements from yfinance and computes credit ratios.
    
    Input: ticker (str) — e.g. "AAPL"
    Output: dict with keys:
        - company_info: dict (name, sector, market cap)
        - ratios: dict (all seven ratios, None if unavailable)
        - raw: dict (raw financial values used in calculations)
        - latest_year: str (fiscal year of latest data)
        - status: dict (flags for what data was available)
    """
    
    result = {
        "company_info": {},
        "ratios": {},
        "raw": {},
        "latest_year": None,
        "status": {}
    }
    
    try:
        stock = yf.Ticker(ticker)
        
        # --- Company Info ---
        try:
            info = stock.info
            result["company_info"] = {
                "name": info.get("longName", "Unknown"),
                "sector": info.get("sector", "Unknown"),
                "industry": info.get("industry", "Unknown"),
                "market_cap": info.get("marketCap", None),
                "description": info.get("longBusinessSummary", "Not available")
            }
        except Exception as e:
            result["status"]["company_info_error"] = str(e)
            result["company_info"] = {
                "name": ticker,
                "sector": "Unknown",
                "industry": "Unknown", 
                "market_cap": None,
                "description": "Not available"
            }

        # --- Financial Statements ---
        try:
            annual_income  = stock.financials
            annual_balance = stock.balance_sheet
            annual_cashflow = stock.cashflow
            latest_year = annual_income.columns[0]
            result["latest_year"] = str(latest_year.date())
        except Exception as e:
            result["status"]["financials_error"] = str(e)
            return result

        # --- Safe Extraction ---
        def safe_get(df, field, col):
            try:
                val = df.loc[field, col]
                return float(val) if not pd.isna(val) else None
            except KeyError:
                return None

        # --- Interest Expense Fallback ---
        def get_interest_expense(income_stmt):
            for col in income_stmt.columns:
                for field in ['Interest Expense', 'Interest Expense Non Operating']:
                    try:
                        val = income_stmt.loc[field, col]
                        if val is not None and not pd.isna(val) and val != 0:
                            return abs(float(val))
                    except KeyError:
                        continue
            return None

        # --- Extract Raw Values ---
        ebit               = safe_get(annual_income, 'EBIT', latest_year)
        ebitda             = safe_get(annual_income, 'EBITDA', latest_year)
        revenue            = safe_get(annual_income, 'Total Revenue', latest_year)
        net_income         = safe_get(annual_income, 'Net Income', latest_year)
        total_debt         = safe_get(annual_balance, 'Total Debt', latest_year)
        net_debt           = safe_get(annual_balance, 'Net Debt', latest_year)
        cash               = safe_get(annual_balance, 'Cash And Cash Equivalents', latest_year)
        total_assets       = safe_get(annual_balance, 'Total Assets', latest_year)
        current_assets     = safe_get(annual_balance, 'Current Assets', latest_year)
        current_liabilities = safe_get(annual_balance, 'Current Liabilities', latest_year)
        working_capital    = safe_get(annual_balance, 'Working Capital', latest_year)
        retained_earnings  = safe_get(annual_balance, 'Retained Earnings', latest_year)
        operating_cf       = safe_get(annual_cashflow, 'Operating Cash Flow', latest_year)
        free_cf            = safe_get(annual_cashflow, 'Free Cash Flow', latest_year)
        interest_expense   = get_interest_expense(annual_income)
        market_cap         = result["company_info"]["market_cap"]

        result["raw"] = {
            "ebit": ebit,
            "ebitda": ebitda,
            "revenue": revenue,
            "net_income": net_income,
            "total_debt": total_debt,
            "net_debt": net_debt,
            "cash": cash,
            "total_assets": total_assets,
            "current_assets": current_assets,
            "current_liabilities": current_liabilities,
            "working_capital": working_capital,
            "retained_earnings": retained_earnings,
            "operating_cf": operating_cf,
            "free_cf": free_cf,
            "interest_expense": interest_expense,
            "market_cap": market_cap
        }

        # --- Compute Ratios ---
        ratios = {}

        # 1. Debt / EBITDA
        ratios["debt_to_ebitda"] = (
            round(total_debt / ebitda, 2) 
            if total_debt and ebitda 
            else None
        )

        # 2. Interest Coverage
        ratios["interest_coverage"] = (
            round(ebit / interest_expense, 2) 
            if ebit and interest_expense 
            else None
        )

        # 3. Current Ratio
        ratios["current_ratio"] = (
            round(current_assets / current_liabilities, 2) 
            if current_assets and current_liabilities 
            else None
        )

        # 4. FCF to Debt
        ratios["fcf_to_debt"] = (
            round(free_cf / total_debt, 2) 
            if free_cf and total_debt 
            else None
        )

        # 5. Net Debt / EBITDA
        ratios["net_debt_to_ebitda"] = (
            round(net_debt / ebitda, 2) 
            if net_debt and ebitda 
            else None
        )

        # 6. DSCR
        ratios["dscr"] = (
            round(operating_cf / total_debt, 2) 
            if operating_cf and total_debt 
            else None
        )

        # 7. Altman Z-Score
        if all([
            working_capital, retained_earnings, ebit,
            market_cap, total_debt, revenue, total_assets
        ]):
            z = (
                1.2 * (working_capital / total_assets) +
                1.4 * (retained_earnings / total_assets) +
                3.3 * (ebit / total_assets) +
                0.6 * (market_cap / total_debt) +
                1.0 * (revenue / total_assets)
            )
            ratios["altman_z"] = round(z, 2)
            ratios["altman_zone"] = (
                "SAFE" if z > 2.99 else 
                "GREY" if z > 1.81 else 
                "DISTRESS"
            )
        else:
            ratios["altman_z"] = None
            ratios["altman_zone"] = None

        result["ratios"] = ratios
        result["status"]["success"] = True

    except Exception as e:
        result["status"]["error"] = str(e)
        result["status"]["success"] = False

    return result


# --- Test it ---
data = get_financial_ratios("AAPL")

print(f"Company: {data['company_info']['name']}")
print(f"Sector: {data['company_info']['sector']}")
print(f"Latest Year: {data['latest_year']}")
print(f"\nRATIOS:")
for k, v in data['ratios'].items():
    print(f"  {k}: {v}")
print(f"\nStatus: {data['status']}")

Company: Apple Inc.
Sector: Technology
Latest Year: 2025-09-30

RATIOS:
  debt_to_ebitda: 0.68
  interest_coverage: 33.83
  current_ratio: 0.89
  fcf_to_debt: 1.0
  net_debt_to_ebitda: 0.43
  dscr: 1.13
  altman_z: 29.77
  altman_zone: SAFE

Status: {'success': True}


In [8]:
# Stress test on two more tickers
# Boeing (BA) — stressed industrial company with real debt concerns
# JPMorgan (JPM) — large cap financial, will test how function handles banks

test_tickers = ["BA", "JPM"]

for ticker in test_tickers:
    print(f"\n{'='*50}")
    print(f"Testing: {ticker}")
    print('='*50)
    
    data = get_financial_ratios(ticker)
    
    print(f"Company: {data['company_info']['name']}")
    print(f"Sector: {data['company_info']['sector']}")
    print(f"Latest Year: {data['latest_year']}")
    print(f"\nRATIOS:")
    for k, v in data['ratios'].items():
        print(f"  {k}: {v}")
    print(f"Status: {data['status']}")
    


Testing: BA
Company: The Boeing Company
Sector: Industrials
Latest Year: 2025-12-31

RATIOS:
  debt_to_ebitda: 7.4
  interest_coverage: 1.95
  current_ratio: 1.19
  fcf_to_debt: -0.03
  net_debt_to_ebitda: 5.83
  dscr: 0.02
  altman_z: 2.89
  altman_zone: GREY
Status: {'success': True}

Testing: JPM
Company: JPMorgan Chase & Co.
Sector: Financial Services
Latest Year: 2025-12-31

RATIOS:
  debt_to_ebitda: None
  interest_coverage: None
  current_ratio: None
  fcf_to_debt: -0.3
  net_debt_to_ebitda: None
  dscr: -0.3
  altman_z: None
  altman_zone: None
Status: {'success': True}


In [10]:
# Add sector exclusion to the function
# We add this check right at the start, before any ratio computation

def get_financial_ratios(ticker: str) -> dict:
    """
    Pulls financial statements from yfinance and computes credit ratios.
    
    Input: ticker (str) — e.g. "AAPL"
    Output: dict with keys:
        - company_info: dict (name, sector, market cap)
        - ratios: dict (all seven ratios, None if unavailable)
        - raw: dict (raw financial values used in calculations)
        - latest_year: str (fiscal year of latest data)
        - status: dict (flags for what data was available)
    """
    
    # Sectors excluded from analysis
    EXCLUDED_SECTORS = [
        "Financial Services",
        "Insurance",
        "Banks",
        "Diversified Financials"
    ]

    result = {
        "company_info": {},
        "ratios": {},
        "raw": {},
        "latest_year": None,
        "status": {}
    }

    try:
        stock = yf.Ticker(ticker)

        # --- Company Info First (needed for sector check) ---
        try:
            info = stock.info
            sector = info.get("sector", "Unknown")
            result["company_info"] = {
                "name": info.get("longName", "Unknown"),
                "sector": sector,
                "industry": info.get("industry", "Unknown"),
                "market_cap": info.get("marketCap", None),
                "description": info.get("longBusinessSummary", "Not available")
            }
        except Exception as e:
            result["status"]["company_info_error"] = str(e)
            result["status"]["success"] = False
            return result

        # --- Sector Check ---
        if sector in EXCLUDED_SECTORS:
            result["status"]["success"] = False
            result["status"]["error"] = (
                f"Sector '{sector}' is not supported. "
                f"This tool is designed for non-financial corporates only. "
                f"Financial institutions use different credit frameworks "
                f"(Tier 1 Capital, NIM, NPL ratio) outside this system's scope."
            )
            return result

        # --- Market Cap Check (large cap only) ---
        market_cap = result["company_info"]["market_cap"]
        if market_cap and market_cap < 10_000_000_000:  # below $10bn
            result["status"]["success"] = False
            result["status"]["error"] = (
                f"Market cap ${market_cap/1e9:.1f}bn is below the $10bn "
                f"large cap threshold. This tool is designed for large cap "
                f"companies only."
            )
            return result

        # --- Financial Statements ---
        try:
            annual_income   = stock.financials
            annual_balance  = stock.balance_sheet
            annual_cashflow = stock.cashflow
            latest_year     = annual_income.columns[0]
            result["latest_year"] = str(latest_year.date())
        except Exception as e:
            result["status"]["financials_error"] = str(e)
            result["status"]["success"] = False
            return result

        # --- Safe Extraction ---
        def safe_get(df, field, col):
            try:
                val = df.loc[field, col]
                return float(val) if not pd.isna(val) else None
            except KeyError:
                return None

        # --- Interest Expense Fallback ---
        def get_interest_expense(income_stmt):
            for col in income_stmt.columns:
                for field in ['Interest Expense', 'Interest Expense Non Operating']:
                    try:
                        val = income_stmt.loc[field, col]
                        if val is not None and not pd.isna(val) and val != 0:
                            return abs(float(val))
                    except KeyError:
                        continue
            return None

        # --- Extract Raw Values ---
        ebit                = safe_get(annual_income, 'EBIT', latest_year)
        ebitda              = safe_get(annual_income, 'EBITDA', latest_year)
        revenue             = safe_get(annual_income, 'Total Revenue', latest_year)
        net_income          = safe_get(annual_income, 'Net Income', latest_year)
        total_debt          = safe_get(annual_balance, 'Total Debt', latest_year)
        net_debt            = safe_get(annual_balance, 'Net Debt', latest_year)
        cash                = safe_get(annual_balance, 'Cash And Cash Equivalents', latest_year)
        total_assets        = safe_get(annual_balance, 'Total Assets', latest_year)
        current_assets      = safe_get(annual_balance, 'Current Assets', latest_year)
        current_liabilities = safe_get(annual_balance, 'Current Liabilities', latest_year)
        working_capital     = safe_get(annual_balance, 'Working Capital', latest_year)
        retained_earnings   = safe_get(annual_balance, 'Retained Earnings', latest_year)
        operating_cf        = safe_get(annual_cashflow, 'Operating Cash Flow', latest_year)
        free_cf             = safe_get(annual_cashflow, 'Free Cash Flow', latest_year)
        interest_expense    = get_interest_expense(annual_income)
        market_cap          = result["company_info"]["market_cap"]

        result["raw"] = {
            "ebit": ebit,
            "ebitda": ebitda,
            "revenue": revenue,
            "net_income": net_income,
            "total_debt": total_debt,
            "net_debt": net_debt,
            "cash": cash,
            "total_assets": total_assets,
            "current_assets": current_assets,
            "current_liabilities": current_liabilities,
            "working_capital": working_capital,
            "retained_earnings": retained_earnings,
            "operating_cf": operating_cf,
            "free_cf": free_cf,
            "interest_expense": interest_expense,
            "market_cap": market_cap
        }

        # --- Compute Ratios ---
        ratios = {}

        ratios["debt_to_ebitda"] = (
            round(total_debt / ebitda, 2)
            if total_debt and ebitda
            else None
        )

        ratios["interest_coverage"] = (
            round(ebit / interest_expense, 2)
            if ebit and interest_expense
            else None
        )

        ratios["current_ratio"] = (
            round(current_assets / current_liabilities, 2)
            if current_assets and current_liabilities
            else None
        )

        ratios["fcf_to_debt"] = (
            round(free_cf / total_debt, 2)
            if free_cf and total_debt
            else None
        )

        ratios["net_debt_to_ebitda"] = (
            round(net_debt / ebitda, 2)
            if net_debt and ebitda
            else None
        )

        ratios["dscr"] = (
            round(operating_cf / total_debt, 2)
            if operating_cf and total_debt
            else None
        )

        if all([
            working_capital, retained_earnings, ebit,
            market_cap, total_debt, revenue, total_assets
        ]):
            z = (
                1.2 * (working_capital / total_assets) +
                1.4 * (retained_earnings / total_assets) +
                3.3 * (ebit / total_assets) +
                0.6 * (market_cap / total_debt) +
                1.0 * (revenue / total_assets)
            )
            ratios["altman_z"]    = round(z, 2)
            ratios["altman_zone"] = (
                "SAFE"     if z > 2.99 else
                "GREY"     if z > 1.81 else
                "DISTRESS"
            )
        else:
            ratios["altman_z"]    = None
            ratios["altman_zone"] = None

        result["ratios"] = ratios
        result["status"]["success"] = True

    except Exception as e:
        result["status"]["error"] = str(e)
        result["status"]["success"] = False

    return result


# --- Test all three cases ---
test_cases = {
    "AAPL": "Should work — large cap tech",
    "JPM":  "Should be blocked — financial sector",
    "BA":   "Should work — stressed industrial",
}

for ticker, description in test_cases.items():
    print(f"\n{'='*50}")
    print(f"{ticker} — {description}")
    print('='*50)
    data = get_financial_ratios(ticker)
    if not data["status"].get("success"):
        print(f"BLOCKED: {data['status'].get('error')}")
    else:
        print(f"Company: {data['company_info']['name']}")
        print(f"Latest Year: {data['latest_year']}")
        for k, v in data["ratios"].items():
            print(f"  {k}: {v}")


AAPL — Should work — large cap tech
Company: Apple Inc.
Latest Year: 2025-09-30
  debt_to_ebitda: 0.68
  interest_coverage: 33.83
  current_ratio: 0.89
  fcf_to_debt: 1.0
  net_debt_to_ebitda: 0.43
  dscr: 1.13
  altman_z: 29.77
  altman_zone: SAFE

JPM — Should be blocked — financial sector
BLOCKED: Sector 'Financial Services' is not supported. This tool is designed for non-financial corporates only. Financial institutions use different credit frameworks (Tier 1 Capital, NIM, NPL ratio) outside this system's scope.

BA — Should work — stressed industrial
Company: The Boeing Company
Latest Year: 2025-12-31
  debt_to_ebitda: 7.4
  interest_coverage: 1.95
  current_ratio: 1.19
  fcf_to_debt: -0.03
  net_debt_to_ebitda: 5.83
  dscr: 0.02
  altman_z: 2.89
  altman_zone: GREY


In [1]:
# Verify the .py file works by importing from it
import importlib
import sys
sys.path.append('..')  # so notebook can find src/

import src.data.financials as fin
importlib.reload(fin)

data = fin.get_financial_ratios("AAPL")
print("Import test:", "PASSED" if data["status"]["success"] else "FAILED")
print("Company:", data["company_info"]["name"])
print("Ratios:", data["ratios"])

Import test: PASSED
Company: Apple Inc.
Ratios: {'debt_to_ebitda': 0.68, 'interest_coverage': 33.83, 'current_ratio': 0.89, 'fcf_to_debt': 1.0, 'net_debt_to_ebitda': 0.43, 'dscr': 1.13, 'altman_z': 29.74, 'altman_zone': 'SAFE'}
